In [1]:
import os

In [3]:
import sys
import json
import numpy as np
import pandas as pd

In [4]:
#! pip install pandas pyarrow fastparquet

In [5]:
import pandas as pd

data_path = '/home/asif/data3/HF_cache/guru_data/train/logic__zebra_puzzle_1.3k.parquet' 
# Load Parquet file
df = pd.read_parquet(data_path)  # or engine='fastparquet'

# Display the dataframe
print(df.head())

   worker_id                     puzzle_id  \
0       2767   puzzle_4x6_level6_instance1   
1       3361  puzzle_3x7_level11_instance7   
2        915  puzzle_3x3_level6_instance24   
3       1192   puzzle_9x3_level4_instance4   
4       4081  puzzle_4x8_level7_instance41   

                                              config  \
0  {'cols': 4, 'instance_id': 1, 'level': 6, 'min...   
1  {'cols': 3, 'instance_id': 7, 'level': 11, 'mi...   
2  {'cols': 3, 'instance_id': 24, 'level': 6, 'mi...   
3  {'cols': 9, 'instance_id': 4, 'level': 4, 'min...   
4  {'cols': 4, 'instance_id': 41, 'level': 7, 'mi...   

                                         instruction  \
0  Solve the following puzzle where you are given...   
1  Solve the following puzzle where you are given...   
2  Solve the following puzzle where you are given...   
3  Solve the following puzzle where you are given...   
4  Solve the following puzzle where you are given...   

                                               cl

In [18]:
#df.all

In [19]:
df.columns

Index(['worker_id', 'puzzle_id', 'config', 'instruction', 'clues',
       'ground_truth', 'data_source', 'prompt', 'ability', 'reward_model',
       'apply_chat_template', 'extra_info', 'qwen2.5_7b_pass_rate',
       'qwen3_30b_pass_rate'],
      dtype='object')

In [20]:
df.head(1)

,worker_id,puzzle_id,config,instruction,clues,ground_truth,data_source,prompt,ability,reward_model,apply_chat_template,extra_info,qwen2.5_7b_pass_rate,qwen3_30b_pass_rate
0,2767,puzzle_4x6_level6_instance1,"{'cols': 4, 'instance_id': 1, 'level': 6, 'min...",Solve the following puzzle where you are given...,[Nationality:chinese is on the left or right o...,"{'header': ['Position', 'Food', 'Job', 'Movie-...",logic__zebra_puzzle_dataset,[{'content': 'Solve the following puzzle where...,logical_reasoning,"{'ground_truth': {'header': ['Position', 'Food...",True,"{'grid_size': '4x6', 'id': '2767', 'raw_input'...",0.0,0.0625


In [23]:
SOLUTION_PROMPT_SYSTEM = """You are an expert logic puzzle solver. You are provided with a logic puzzle.

Your task is to:
1. Analyze the clues step by step.
2. Derive a correct final solution.
3. Return the result STRICTLY as a single valid JSON object.

CRITICAL FORMAT REQUIREMENTS:
- Output ONLY a JSON object, NO natural language, NO markdown, NO code fences.
- The top-level JSON MUST have exactly two keys: "reasoning" and "solution".
- "reasoning" MUST be a SHORT English explanation (1–5 sentences, not more).
- "solution" MUST be an object with:
  - "header": a list of column names (e.g. ["House", "Name", "Pet", "..."])
  - "rows": a list of rows, where each row is a list of strings, one per column.

Example of the REQUIRED SHAPE (this is ONLY an example, not the answer):

{
  "reasoning": "Your step-by-step logic here, but concise.",
  "solution": {
    "header": ["House", "Name", "Pet", "..."],
    "rows": [
      ["1", "Eric", "cat", "..."],
      ["2", "Arnold", "dog", "..."]
    ]
  }
}

Do NOT include any text before or after the JSON.
"""

SOLUTION_PROMPT_USER = """PUZZLE:
{puzzle}

Please provide your reasoning and solution:"""


## Read and check json file

In [8]:
data_path = '/home/asif/data3/HF_cache/ZebraLogic/Zebra_Puzzle_small_320.json'



In [9]:
with open(data_path, 'r') as f:
    content = json.load(f)
    if isinstance(content, list):
        data = content

In [13]:
data[0].keys()

dict_keys(['id', 'size', 'puzzle', 'solution', 'created_at'])

In [14]:
data[0]

{'id': 'lgp-test-2x2-33',
 'size': '2*2',
 'puzzle': 'There are 2 houses, numbered 1 to 2 from left to right, as seen from across the street. Each house is occupied by a different person. Each house has a unique attribute for each of the following characteristics:\n - Each person has a unique name: `Eric`, `Arnold`\n - Each person has a unique type of pet: `dog`, `cat`\n\n## Clues:\n1. Eric is somewhere to the left of Arnold.\n2. The person who owns a dog is not in the first house.\n',
 'solution': {'header': ['House', 'Name', 'Pet'],
  'rows': [['1', 'Eric', 'cat'], ['2', 'Arnold', 'dog']]},
 'created_at': '2024-07-03T21:21:29.204640'}

In [29]:
data_processed = []

for line in data:
    entry = {}
    
    entry['instruction'] = SOLUTION_PROMPT_SYSTEM + SOLUTION_PROMPT_USER.format(puzzle=line['puzzle'])
    entry['apply_chat_template'] = False
    entry['puzzle_id'] = line['id']
    entry['ground_truth'] = line['solution']
    entry['reward_model'] = line['solution']
    entry['config'] = line['size']
    entry['extra_info'] = line['created_at']
    
    data_processed.append(entry)
    

In [30]:
len(data_processed)

320

In [31]:
out_path = '/home/asif/data3/HF_cache/ZebraPuzzle_guru/zebra_puzzle.json'

with open(out_path, "w", encoding="utf-8") as f:
    for item in data_processed:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")